# Predicting Heart Disease- A Machine Learning Approach to Health

---

In this notebook, we will analyze a dataset of patients' metabolic information to see if this data can help with indications as to whether or not an individual has heart disease. 

According to the Mayo Clinic-

>Heart disease describes a range of conditions that affect the heart.
>Heart disease includes:
>Blood vessel disease, such as coronary artery disease.
>Irregular heartbeats, called arrhythmias.
>Heart conditions that you're born with, called congenital heart defects.
>Disease of the heart muscle.
>Heart valve disease.
>Many forms of heart disease can be prevented or treated with healthy lifestyle choices.
---

The dataset features are as follows-

<table>
  <tbody>
    <tr> 
    <td>Feature</td>
      <td>🧓 Age</td>
      <td>🚹 Sex</td>
      <td>💔 Chest pain type</td>
      <td>💉 BP</td>
      <td>🧈 Cholesterol</td>
        <td>🍬 FBS over 120</td>
        <td>📈 EKG results</td>
        <td>❤️ Max HR</td>
        <td>🏃 Exercise angina</td>
        <td>📉 ST depression</td>
        <td>⛰️ Slope of ST</td>
        <td>🩸 Number of vessels fluro</td>
        <td>🧬 Thallium</td>
        <td>🎯 Heart Disease</td>
    </tr>
    <tr>
        <td>Description</td>
      <td>Age of the patient (in years)</td>
      <td>Gender of the patient (1 = Male, 0 = Female)</td>
      <td>Type of chest pain</td>
     <td>Resting blood pressure (mm Hg)</td>
    <td>Serum cholesterol level (mg/dL)</td>
    <td>Fasting blood sugar > 120 mg/dL (1 = True, 0 = False)</td>
    <td>Resting electrocardiogram results</td>
    <td>Maximum heart rate achieved</td>
    <td>Exercise-induced angina (1 = Yes, 0 = No)</td>
    <td>ST depression induced by exercise relative to rest</td>
        <td>Slope of the peak exercise ST segment</td>
    <td>Number of major vessels (0–3) colored by fluoroscopy</td>
    <td>Thallium stress test result (categorical medical indicator)</td>
    <td>Presence of Heart Disease</td>
    </tr>
  </tbody>
</table>

📊 Overall Feature Design Strengths-

| Domain           | Represented               |
| ---------------- | ------------------------- |
| Demographics     | Age, Sex                  |
| Risk Factors     | BP, Cholesterol, FBS      |
| Symptoms         | Chest pain, angina        |
| Functional Tests | Stress ECG, ST depression |
| Imaging          | Fluoroscopy, Thallium     |
| Fitness          | Max HR                    |

---

🫀 Clinical & Predictive Rationale for Each Feature

This dataset is essentially a compact cardiovascular risk profiling system. Every variable reflects a known contributor to coronary artery disease (CAD) and myocardial ischemia.

With that being said- lets get started!

In [ ]:
# Import libraries
import numpy as np 
import pandas as pd 
from IPython.display import display, HTML
import os
from sklearn.model_selection import train_test_split
import xgboost as xgb
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import catboost as cb

In [ ]:
# Get file information from folder
for dirname, _, filenames in os.walk('/kaggle/input'):

    for filename in filenames:
        
        print(os.path.join(dirname, filename))

In [ ]:
# Read the data
df = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv').drop('id', axis = 1)

In [ ]:
# Read the data
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv') #.drop('id', axis = 1)

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df['Heart Disease'].value_counts()

In [ ]:
df.isnull().sum()

In [ ]:
# Map the outcome to numeric
outcomes = {'Absence': 0, 'Presence': 1}
df['Heart Disease'] = df['Heart Disease'].map(outcomes)

In [ ]:
# Split the data for training
x = df[[i for i in df.columns if i != 'Heart Disease']]
y = df['Heart Disease']

xtrain, xval, ytrain, yval = train_test_split(x, y, test_size = .2, random_state = 100)

In [ ]:
xtrain.head()

In [ ]:
ytrain.head()

In [ ]:
# Build a baseline randomf forest model
rf = RandomForestClassifier(random_state = 100)
rf.fit(xtrain, ytrain)

In [ ]:
# Get performance
preds = rf.predict(xval)
print(f"Random Forest Classifier performance- {round(roc_auc_score(preds, yval), 3)}")

In [ ]:
# Submit the test set for kaggle grading
rftest = rf.predict(test.drop('id', axis = 1))
test['Heart Disease'] = rftest

In [ ]:
test[['id', 'Heart Disease']].to_csv('heart_disease_test_rf.csv', index = False)

In [ ]:
# Define kaggle credentials
os.environ['KAGGLE_USERNAME'] = "deeguy"
os.environ['KAGGLE_KEY'] = "KGAT_f17b145a2b548a205d66ca22b4030fcb"

In [ ]:
!kaggle competitions submit -c playground-series-s6e2 -f heart_disease_test_rf.csv -m "rf heart disease predictions test set"

In [ ]:
# Try XGBoost model
xgb_clf = xgb.XGBClassifier(random_state = 100)
xgb_clf.fit(xtrain, ytrain)

In [ ]:
# Get performance
xgb_preds = xgb_clf.predict(xval)
print(f"XGBoost Classifier performance- {round(roc_auc_score(xgb_preds, yval), 3)}")

In [ ]:
# Submit the test set for kaggle grading
xgtest = xgb_clf.predict(test.drop(['id', 'Heart Disease'], axis = 1))
test['Heart Disease'] = xgtest

# Write to file
test[['id', 'Heart Disease']].to_csv('heart_disease_test_xgb.csv', index = False)

In [ ]:
!kaggle competitions submit -c playground-series-s6e2 -f heart_disease_test_xgb.csv -m "xgb heart disease predictions test set"

In [ ]:
# Define the objective function for Optuna
def objective(trial):

    # Outline parameters for testing
    params = {'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
              'depth': trial.suggest_int('depth', 3, 10),
              'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0),
              'iterations': 1000,
              'random_state': 100,
              'verbose': 0, 
              'eval_metric': 'AUC',
              'early_stopping_rounds': 10}

    # Train model
    model = cb.CatBoostClassifier(**params)
    model.fit(xtrain, ytrain, eval_set = (xval, yval))
    
    # Get the best AUC score achieved during training on the test set
    best_auc = round(model.get_best_score()['validation']['AUC'], 5)

    return best_auc

In [ ]:
# Create a study and optimize
study = optuna.create_study(direction = 'maximize') 
study.optimize(objective, n_trials = 50, n_jobs = 4) 

# Print the best hyperparameters and AUC
print(f"Best trial value (AUC): {study.best_value}")
print(f"Best hyperparameters: {study.best_params}")

In [ ]:
# Isolate max performance
best_trial = study.best_trial
print("Best Params:", best_trial.params)

In [ ]:
# Build the catboost model with optimized params
catboost_model = cb.CatBoostClassifier(**best_trial.params)
catboost_model.fit(xtrain, ytrain)

#catboost_model = cb.CatBoostClassifier(learning_rate = 0.16276813510129443, 
#                                       depth = 3, 
#                                       l2_leaf_reg = 6.185417265799672, 
#                                       iterations = 1000,
#                                       random_state = 100,
#                                       eval_metric = 'AUC',
#                                       early_stopping_rounds = 100,
#                                       verbose = False)

#catboost_model.fit(xtrain, ytrain, eval_set = (xval, yval))

In [ ]:
# Get performance
cb_preds = catboost_model.predict(xval)
print(f"Cat Boost Classifier performance- {round(roc_auc_score(cb_preds, yval), 3)}")

In [ ]:
# Submit the test set for kaggle grading
cbtest = catboost_model.predict(test.drop(['id', 'Heart Disease'], axis = 1))
test['Heart Disease'] = cbtest

# Write to file
test[['id', 'Heart Disease']].to_csv('heart_disease_test_catboost.csv', index = False)

In [ ]:
!kaggle competitions submit -c playground-series-s6e2 -f heart_disease_test_catboost.csv -m "catboost heart disease predictions test set"

In [ ]:
# Define objective function for optuna with lgith gbm
def objective(trial):

    dtrain = lgb.Dataset(xtrain, label = ytrain)
    
    params = {"objective": "binary",
              "metric": "binary_logloss",
              "verbosity": -1,
              "boosting_type": "gbdt",
              "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
              "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
              "num_leaves": trial.suggest_int("num_leaves", 2, 256),
              "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
              "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
              "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
              "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),}

        
    gbm = lgb.train(params, dtrain)
    preds = gbm.predict(xval)
    pred_labels = np.rint(preds)
    auc = roc_auc_score(yval, pred_labels)

    return auc

In [ ]:
# Create a study and optimize
study = optuna.create_study(direction = 'maximize') 
study.optimize(objective, n_trials = 50, n_jobs = 4) 

# Print the best hyperparameters and AUC
print('Light GBM Model- ')
print(f"Best trial value (AUC): {study.best_value}")
print(f"Best hyperparameters: {study.best_params}")

In [ ]:
dtrain = lgb.Dataset(xtrain, label = ytrain)
gbm = lgb.train(study.best_params, dtrain)
gbmtest = gbm.predict(test.drop(['id', 'Heart Disease'], axis = 1))

test['Heart Disease'] = gbmtest

# Write to file
test[['id', 'Heart Disease']].to_csv('heart_disease_test_gbm.csv', index = False)

In [ ]:
!kaggle competitions submit -c playground-series-s6e2 -f heart_disease_test_gbm.csv -m "LightGBM heart disease predictions test set"

In [ ]:
!kaggle competitions submissions -c playground-series-s6e2